In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- load data ---
DATA_DIR = Path("../data/raw")
train = pd.read_csv(DATA_DIR / "train.csv")

target = "SalePrice"
id_col = "Id"

y = train[target]
X = train.drop(columns=[target, id_col])

# --- local split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- feature groups ---
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# --- preprocessing ---
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# --- final candidate ---
hgb = HistGradientBoostingRegressor(
    l2_regularization=0.1,
    learning_rate=0.1,
    max_leaf_nodes=15,
    random_state=42
)

final_model = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", preprocessor),
        ("model", hgb)
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

# --- fit on X_train ---
final_model.fit(X_train, y_train)

# --- evaluate on X_test ---
y_pred = final_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
rmsle = np.sqrt(mean_squared_error(np.log1p(y_test), np.log1p(np.maximum(y_pred,0))))

print(f"Final test metrics:")
print(f"MAE = {mae:.2f}")
print(f"RMSE = {rmse:.2f}")
print(f"R² = {r2:.4f}")
print(f"RMSLE = {rmsle:.4f}")

C:\Users\Александр\AppData\Local\Temp\ipykernel_24456\2465993873.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()


Final test metrics:
MAE = 15830.01
RMSE = 27974.15
R² = 0.8980
RMSLE = 0.1332


- Test metrics vs CV expectations
- Signs of overfitting/instability
- Final honest model quality
- Limitations of RMSE-priority choice
- Note: GradientBoostingRegressor remains an alternative by MAE/RMSLE but was not evaluated